In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_excel('capstone_airline_reviews3.xlsx')
print(df.shape)
print(df.columns.tolist())
df.head()

In [ ]:
before = len(df)
df = df.dropna(how='all').reset_index(drop=True)
print(f"Dropped {before - len(df)} fully-blank rows")

missing_pct = df.isnull().mean()
cols_to_drop = missing_pct[missing_pct > 0.9].index.tolist()
df = df.drop(columns=cols_to_drop)

cat_cols = df.select_dtypes(include='object').columns
df[cat_cols] = df[cat_cols].fillna('Unknown')

num_cols = df.select_dtypes(include='number').columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

print(df.isnull().sum().sum(), "missing values remain ")

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-z0-9\s.,!?]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['review_clean'] = df['customer_review'].apply(clean_text)

In [ ]:
before = len(df)
df = df.drop_duplicates()
print(f"Dropped {before - len(df)} duplicate rows")
df.to_csv('airline_reviews_clean.csv', index=False)
print(f"Final shape: {df.shape}")

In [ ]:
from textblob import TextBlob

def get_sentiment(text):
    return TextBlob(str(text)).sentiment.polarity

df['sentiment_score'] = df['review_clean'].apply(get_sentiment)

df.to_csv('airline_reviews_with_sentiment.csv', index=False)
print("Sentiment scoring done.")

In [ ]:
df = pd.read_csv('airline_reviews_with_sentiment.csv')
df = df[df['recommended'].astype(str).str.strip().str.lower().isin(['yes', 'no'])].copy()
print(df['recommended'].value_counts())

In [ ]:
from sklearn.model_selection import train_test_split

feature_cols = ['seat_comfort', 'cabin_service', 'food_bev', 'entertainment',
                 'ground_service', 'value_for_money', 'traveller_type', 'cabin']

X = df[feature_cols].copy()
X = pd.get_dummies(X, columns=['traveller_type', 'cabin'], drop_first=True)

y = df['recommended'].astype(str).str.strip().str.lower().map({'no': 0, 'yes': 1})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)

y_pred_log = log_model.predict(X_test)
y_proba_log = log_model.predict_proba(X_test)[:, 1]

print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_log))

In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42, eval_metric='logloss')
xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)
y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

print("XGBoost Accuracy:", accuracy_score(y_test, y_pred_xgb))

In [ ]:
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'XGBoost'],
    'Accuracy': [accuracy_score(y_test, y_pred_log), accuracy_score(y_test, y_pred_xgb)],
    'Precision': [precision_score(y_test, y_pred_log), precision_score(y_test, y_pred_xgb)],
    'Recall': [recall_score(y_test, y_pred_log), recall_score(y_test, y_pred_xgb)],
    'F1': [f1_score(y_test, y_pred_log), f1_score(y_test, y_pred_xgb)],
    'ROC-AUC': [roc_auc_score(y_test, y_proba_log), roc_auc_score(y_test, y_proba_xgb)],
})
print(results)

In [ ]:
df = pd.read_csv('airline_reviews_with_sentiment.csv')

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

rating_cols = ['seat_comfort', 'cabin_service', 'food_bev', 'entertainment',
               'ground_service', 'value_for_money']
X_cluster = df[rating_cols].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

scores = {}
for k_test in range(2, 8):
    km = KMeans(n_clusters=k_test, random_state=42, n_init=10).fit(X_scaled)
    scores[k_test] = silhouette_score(X_scaled, km.labels_, sample_size=5000, random_state=42)

print("Silhouette scores by k:", scores)

k = max(scores, key=scores.get)
print(f"Best k: {k}")

kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_scaled)
print(df['cluster'].value_counts())

In [ ]:
united_negative = df[
    (df['airline'].str.contains('united', case=False, na=False)) &
    (df['sentiment_score'] < 0)
]['review_clean'].sample(n=30, random_state=42).tolist()

sample_text = "\n\n".join(united_negative)

prompt = f"""Analyze these United Airlines customer complaints. Identify:
1. The top 3 recurring complaint themes
2. Which customer segment seems most affected (business/leisure/family)
3. One specific, actionable recommendation per theme

Reviews:
{sample_text}
"""

print(prompt[:500])

In [ ]:
!pip install -U -q google-generativeai

In [ ]:
from google.colab import userdata
import os
os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')

In [ ]:
import google.generativeai as genai
from google.api_core import client_options

options = client_options.ClientOptions(
    api_endpoint="generativelanguage.googleapis.com"
)

genai.configure(
    api_key=os.environ["GEMINI_API_KEY"],
    transport="grpc",
    client_options=options,
)

model = genai.GenerativeModel("gemini-3.6-flash")
llm_response = model.generate_content(prompt, request_options={"timeout": 180})
llm_themes = llm_response.text

print(f"Total reviews analyzed: {len(df)}")
print(f"United reviews: {(df['airline'].str.contains('united', case=False, na=False)).sum()}")
print(f"XGBoost accuracy: {results.loc[results['Model'] == 'XGBoost', 'Accuracy'].values[0]:.4f}")
print(f"Number of customer clusters found: {k}")
print("\nLLM-identified complaint themes and recommendations:")
print(llm_themes)

In [ ]:
top_competitors = df['airline'].value_counts().head(4).index.tolist()
comparison = df[df['airline'].isin(top_competitors)].groupby('airline')[
    rating_cols + ['overall', 'sentiment_score']
].mean().round(2)

print(comparison)
comparison.to_csv('competitor_comparison.csv')